# Brain Tumor MRI Image Classification (Simplified)

Sections follow the required workflow:

1. Dataset Loading  
2. Data Preprocessing  
3. Custom CNN Model  
4. Transfer Learning Models (ResNet50, MobileNetV2, InceptionV3, EfficientNetB0)  
5. Model Training  
6. Model Evaluation  
7. Model Comparison  
8. Model Saving (.h5)


In [9]:
!pip install tensorflow scikit-learn pillow


## Import Libraries


In [10]:
import os
import numpy as np
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from tensorflow.keras.applications import ResNet50, MobileNetV2, InceptionV3, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess

from sklearn.metrics import classification_report, confusion_matrix


## Dataset Loading (Grayscale Images)


In [12]:
import os

DATA_DIR = r"C:\Users\Kalki Prathisha\Downloads\SkillDevelopment\AIML_Projects\Brain_Tumor_MRI_Image_Classification\Tumour"   

TRAIN_DIR = DATA_DIR + "/train"
VALID_DIR = DATA_DIR + "/valid"    
TEST_DIR  = DATA_DIR + "/test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [13]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    color_mode = "grayscale",
    label_mode = "int",
    shuffle = True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VALID_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    color_mode = "grayscale",
    label_mode = "int",
    shuffle = False
)

test_ds  = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size = IMG_SIZE,
    batch_size = BATCH_SIZE,
    color_mode = "grayscale",
    label_mode = "int",
    shuffle = False
)

class_names = train_ds.class_names
num_classes = len(class_names)

print("Class Names:", class_names)


Found 1695 files belonging to 4 classes.
Found 502 files belonging to 4 classes.
Found 246 files belonging to 4 classes.
Class Names: ['glioma', 'meningioma', 'no_tumor', 'pituitary']


## Data Preprocessing


In [14]:
normalization = layers.Rescaling(1./255)

def convert_to_rgb(x):
    return tf.image.grayscale_to_rgb(x)


## Custom CNN Model


In [15]:
custom_model = models.Sequential([
    
    layers.Input(shape=(224,224,1)),
    
    normalization,
    
    layers.Conv2D(32,3,activation="relu"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(64,3,activation="relu"),
    layers.MaxPooling2D(),
    
    layers.Conv2D(128,3,activation="relu"),
    layers.MaxPooling2D(),
    
    layers.Flatten(),
    
    layers.Dense(128,activation="relu"),
    layers.Dropout(0.3),
    
    layers.Dense(num_classes,activation="softmax")
])

custom_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

custom_model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 1)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,168,900 (42.61 MB)

 Trainable params: 11,168,900 (42.61 MB)

 Non-trainable params: 0 (0.00 B)

## Train Custom CNN


In [16]:
callbacks=[
    EarlyStopping(patience=5,restore_best_weights=True),
    ModelCheckpoint("custom_model.h5",save_best_only=True)
]

history_custom = custom_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks
)


Epoch 1/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 256ms/step - accuracy: 0.5146 - loss: 1.2615

53/53 ━━━━━━━━━━━━━━━━━━━━ 16s 284ms/step - accuracy: 0.6171 - loss: 0.9677 - val_accuracy: 0.7072 - val_loss: 0.7248
Epoch 2/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 272ms/step - accuracy: 0.7984 - loss: 0.5462

53/53 ━━━━━━━━━━━━━━━━━━━━ 16s 306ms/step - accuracy: 0.8118 - loss: 0.5117 - val_accuracy: 0.8207 - val_loss: 0.5218
Epoch 3/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 18s 341ms/step - accuracy: 0.8608 - loss: 0.3822 - val_accuracy: 0.7908 - val_loss: 0.5233
Epoch 4/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step - accuracy: 0.8663 - loss: 0.3224

53/53 ━━━━━━━━━━━━━━━━━━━━ 18s 341ms/step - accuracy: 0.8903 - loss: 0.2762 - val_accuracy: 0.8685 - val_loss: 0.3909
Epoch 5/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 18s 335ms/step - accuracy: 0.9351 - loss: 0.1968 - val_accuracy: 0.8705 - val_loss: 0.4447
Epoch 6/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 302ms/step - accuracy: 0.9535 - loss: 0.1499

53/53 ━━━━━━━━━━━━━━━━━━━━ 18s 334ms/step - accuracy: 0.9475 - loss: 0.1418 - val_accuracy: 0.8964 - val_loss: 0.3790
Epoch 7/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 17s 328ms/step - accuracy: 0.9634 - loss: 0.0991 - val_accuracy: 0.9084 - val_loss: 0.3961
Epoch 8/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 17s 327ms/step - accuracy: 0.9835 - loss: 0.0610 - val_accuracy: 0.9203 - val_loss: 0.4108
Epoch 9/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 18s 331ms/step - accuracy: 0.9687 - loss: 0.0788 - val_accuracy: 0.8865 - val_loss: 0.4458
Epoch 10/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 18s 332ms/step - accuracy: 0.9794 - loss: 0.0541 - val_accuracy: 0.9143 - val_loss: 0.4902
Epoch 11/20
53/53 ━━━━━━━━━━━━━━━━━━━━ 17s 327ms/step - accuracy: 0.9864 - loss: 0.0358 - val_accuracy: 0.9163 - val_loss: 0.4899


## Transfer Learning Model Builder


In [17]:
def build_transfer_model(base_model, preprocess):

    base_model.trainable=False

    inputs = layers.Input(shape=(224,224,1))
    
    x = layers.Lambda(convert_to_rgb)(inputs)
    x = preprocess(x)
    
    x = base_model(x,training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128,activation="relu")(x)
    
    outputs = layers.Dense(num_classes,activation="softmax")(x)

    model = tf.keras.Model(inputs,outputs)

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model


### ResNet50


In [18]:
resnet_model = build_transfer_model(
    ResNet50(weights="imagenet",include_top=False,input_shape=(224,224,3)),
    resnet_preprocess
)

resnet_model.fit(train_ds,validation_data=val_ds,epochs=10)
resnet_model.save("resnet50_model.h5")


94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 11s 0us/step



Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 47s 828ms/step - accuracy: 0.8094 - loss: 0.5323 - val_accuracy: 0.9064 - val_loss: 0.2989
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 955ms/step - accuracy: 0.9263 - loss: 0.1970 - val_accuracy: 0.9243 - val_loss: 0.2323
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 51s 966ms/step - accuracy: 0.9504 - loss: 0.1380 - val_accuracy: 0.8825 - val_loss: 0.3194
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 51s 961ms/step - accuracy: 0.9516 - loss: 0.1349 - val_accuracy: 0.9402 - val_loss: 0.2397
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 952ms/step - accuracy: 0.9752 - loss: 0.0808 - val_accuracy: 0.9303 - val_loss: 0.2309
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 948ms/step - accuracy: 0.9829 - loss: 0.0628 - val_accuracy: 0.9263 - val_loss: 0.2294
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 51s 962ms/step - accuracy: 0.9847 - loss: 0.0518 - val_accuracy: 0.9283 - val_loss: 0.2205
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 50s 953ms/step - accuracy: 0.9923 - loss: 0.0292 - val_accu

### MobileNetV2


In [19]:
mobilenet_model = build_transfer_model(
    MobileNetV2(weights="imagenet",include_top=False,input_shape=(224,224,3)),
    mobilenet_preprocess
)

mobilenet_model.fit(train_ds,validation_data=val_ds,epochs=10)
mobilenet_model.save("mobilenet_model.h5")


Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 16s 261ms/step - accuracy: 0.7965 - loss: 0.5338 - val_accuracy: 0.8705 - val_loss: 0.3812
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 13s 255ms/step - accuracy: 0.9204 - loss: 0.2235 - val_accuracy: 0.8765 - val_loss: 0.3406
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 13s 248ms/step - accuracy: 0.9510 - loss: 0.1460 - val_accuracy: 0.8725 - val_loss: 0.4043
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 15s 278ms/step - accuracy: 0.9475 - loss: 0.1339 - val_accuracy: 0.9203 - val_loss: 0.2541
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 15s 278ms/step - accuracy: 0.9717 - loss: 0.0911 - val_accuracy: 0.8805 - val_loss: 0.2912
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 15s 287ms/step - accuracy: 0.9912 - loss: 0.0504 - val_accuracy: 0.9143 - val_loss: 0.2371
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 14s 271ms/step - accuracy: 0.9912 - loss: 0.0446 - val_accuracy: 0.9203 - val_loss: 0.2992
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 15s 276ms/step - accuracy: 0.9953 - loss: 0.0356 - val_accu

### InceptionV3


In [20]:
inception_model = build_transfer_model(
    InceptionV3(weights="imagenet",include_top=False,input_shape=(224,224,3)),
    inception_preprocess
)

inception_model.fit(train_ds,validation_data=val_ds,epochs=10)
inception_model.save("inception_model.h5")


87910968/87910968 ━━━━━━━━━━━━━━━━━━━━ 10s 0us/step
Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 30s 492ms/step - accuracy: 0.7068 - loss: 0.7488 - val_accuracy: 0.8267 - val_loss: 0.4742
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 29s 548ms/step - accuracy: 0.8720 - loss: 0.3564 - val_accuracy: 0.8287 - val_loss: 0.4789
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 29s 558ms/step - accuracy: 0.9027 - loss: 0.2600 - val_accuracy: 0.8665 - val_loss: 0.3842
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 29s 541ms/step - accuracy: 0.9339 - loss: 0.2033 - val_accuracy: 0.8685 - val_loss: 0.3859
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 771s 15s/step - accuracy: 0.9516 - loss: 0.1489 - val_accuracy: 0.8526 - val_loss: 0.4448
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 22s 408ms/step - accuracy: 0.9593 - loss: 0.1314 - val_accuracy: 0.8904 - val_loss: 0.3656
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 29s 557ms/step - accuracy: 0.9628 - loss: 0.1171 - val_accuracy: 0.8745 - val_loss: 0.4270
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 30s 559m

### EfficientNetB0


In [21]:
efficientnet_model = build_transfer_model(
    EfficientNetB0(weights="imagenet",include_top=False,input_shape=(224,224,3)),
    efficientnet_preprocess
)

efficientnet_model.fit(train_ds,validation_data=val_ds,epochs=10)
efficientnet_model.save("efficientnet_model.h5")


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 25s 376ms/step - accuracy: 0.7829 - loss: 0.5687 - val_accuracy: 0.8645 - val_loss: 0.3640
Epoch 2/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 22s 407ms/step - accuracy: 0.9044 - loss: 0.2617 - val_accuracy: 0.8904 - val_loss: 0.2959
Epoch 3/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 406ms/step - accuracy: 0.9186 - loss: 0.2188 - val_accuracy: 0.9084 - val_loss: 0.2655
Epoch 4/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 395ms/step - accuracy: 0.9457 - loss: 0.1669 - val_accuracy: 0.8884 - val_loss: 0.3236
Epoch 5/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 389ms/step - accuracy: 0.9516 - loss: 0.1419 - val_accuracy: 0.9163 - val_loss: 0.2406
Epoch 6/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 394ms/step - accuracy: 0.9593 - loss: 0.1317 - val_accuracy: 0.8984 - val_loss: 0.2748
Epoch 7/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 396ms/step - accuracy: 0.9676 - loss: 0.1099 - val_accuracy: 0.9223 - val_loss: 0.2272
Epoch 8/10
53/53 ━━━━━━━━━━━━━━━━━━━━ 21s 392m

## Model Evaluation


In [25]:
def evaluate(model, name):

    y_true = []
    y_pred = []

    for images, labels in test_ds:

        preds = model.predict(images)
        preds = np.argmax(preds, axis=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print("\n", name)
    print(classification_report(y_true, y_pred, target_names=class_names))

    accuracy = np.mean(y_true == y_pred)

    print("Accuracy:", accuracy)

    return accuracy

In [26]:
results = {}

results["Custom CNN"] = evaluate(custom_model, "Custom CNN")
results["ResNet50"] = evaluate(resnet_model, "ResNet50")
results["MobileNetV2"] = evaluate(mobilenet_model, "MobileNetV2")
results["InceptionV3"] = evaluate(inception_model, "InceptionV3")
results["EfficientNetB0"] = evaluate(efficientnet_model, "EfficientNetB0")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 102ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 106ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step

 Custom CNN
              precision    recall  f1-score   support

      glioma       0.94      0.84      0.89        80
  meningioma       0.72      0.84      0.77        63
    no_tumor       0.80      0.82      0.81        49
   pituitary       1.00      0.94      0.97        54

    accuracy                           0.86       246
   macro avg       0.86      0.86      0.86       246
weighted avg       0.87      0.86      0.86       246

Accuracy: 0.8577235772357723
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 548ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 570ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 549ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 542ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 547ms/step
1/1 ━━━━━━━━━━━━━━━━

In [27]:
best_model_name = max(results, key=results.get)

print("\nModel Comparison Results:")
print(results)

print("\nBest Model:", best_model_name)
print("Best Accuracy:", results[best_model_name])


Model Comparison Results:
{'Custom CNN': np.float64(0.8577235772357723), 'ResNet50': np.float64(0.926829268292683), 'MobileNetV2': np.float64(0.9349593495934959), 'InceptionV3': np.float64(0.9065040650406504), 'EfficientNetB0': np.float64(0.9308943089430894)}

Best Model: MobileNetV2
Best Accuracy: 0.9349593495934959


In [28]:
model_map = {
    "Custom CNN": custom_model,
    "ResNet50": resnet_model,
    "MobileNetV2": mobilenet_model,
    "InceptionV3": inception_model,
    "EfficientNetB0": efficientnet_model
}

best_model = model_map[best_model_name]

best_model.save("best_brain_tumor_model.h5")

print("Best model saved as best_brain_tumor_model.h5")

Best model saved as best_brain_tumor_model.h5
